# RL Post-Training GPU Memory Profile (TRL track)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rahulsanjay009/rl-memory-profiling/blob/main/notebooks/01_trl_baseline_profile.ipynb)

Runs a small PPO loop (TRL + Qwen2.5-0.5B-Instruct) on a single GPU, tags each step's rollout and update phases, and measures:

1. the memory-vs-time profile across phases,
2. a parameter / gradient / optimizer-state breakdown at representative checkpoints, and
3. an idle-time estimate per category.

See the repo [README](https://github.com/rahulsanjay009/rl-memory-profiling) for the full plan and cited related work.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q trl transformers torch pandas matplotlib
    if not os.path.exists("rl-memory-profiling"):
        !git clone -q https://github.com/rahulsanjay009/rl-memory-profiling.git
    sys.path.append("rl-memory-profiling/scripts")
else:
    sys.path.append("../scripts")

from memory_poller import MemoryPoller, compute_category_breakdown

## 1. Load model and PPO trainer

`AutoModelForCausalLMWithValueHead` / `PPOTrainer` are TRL's standard PPO classes. If your installed TRL version has renamed these (TRL's API has shifted a few times), adjust the imports accordingly — the profiling code below doesn't depend on which TRL trainer class you use.

In [ ]:
import torch
from transformers import AutoTokenizer
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME).cuda()
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME).cuda()

config = PPOConfig(batch_size=4, mini_batch_size=2)
trainer = PPOTrainer(config, model, ref_model, tokenizer)
optimizer = getattr(trainer, "optimizer", None)  # adjust if your TRL version exposes it differently

## 2. Run steps with phase-tagged memory polling (Experiments 1–2)

Each step wraps generation in `poller.phase("rollout")` and the policy update in `poller.phase("update")`, and captures a parameter/gradient/optimizer-state breakdown right before and right after the update.

In [ ]:
N_STEPS = 20
prompt = "Explain why the sky is blue."
query_tensor = tokenizer(prompt, return_tensors="pt").input_ids.cuda()

poller = MemoryPoller(interval_s=0.05)
poller.start()

breakdowns = []  # (step, checkpoint_label, category_bytes_dict)

try:
    for step in range(N_STEPS):
        with poller.phase("rollout"):
            response_tensor = trainer.generate(query_tensor, max_new_tokens=64)

        breakdowns.append((step, "start_of_update", compute_category_breakdown(model, optimizer)))

        reward = torch.tensor([1.0]).cuda()  # placeholder reward signal
        with poller.phase("update"):
            trainer.step([query_tensor[0]], [response_tensor[0]], [reward])

        breakdowns.append((step, "peak_of_update", compute_category_breakdown(model, optimizer)))
finally:
    poller.stop()

df = poller.to_dataframe()
os.makedirs("../results", exist_ok=True)
poller.to_csv("../results/trl_baseline_trace.csv")
df.tail()

## 3. Baseline phase memory profile (Experiment 1)

In [ ]:
import matplotlib.pyplot as plt

PHASE_COLORS = {"rollout": "#4C72B0", "update": "#DD8452", "idle": "#B0B0B0"}

df["allocated_gb"] = df["allocated_bytes"] / 1e9

fig, ax = plt.subplots(figsize=(10, 4))
for phase, group in df.groupby("phase"):
    ax.scatter(group["t"], group["allocated_gb"], s=4, label=phase, color=PHASE_COLORS.get(phase, "#333"))
ax.set_xlabel("time (s)")
ax.set_ylabel("GPU memory allocated (GB)")
ax.set_title(f"{MODEL_NAME} — PPO memory over rollout/update phases")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Memory category breakdown (Experiment 2)

In [ ]:
import pandas as pd

bdf = pd.DataFrame(
    [{"step": s, "checkpoint": c, **{k: v / 1e9 for k, v in d.items()}} for s, c, d in breakdowns]
)
avg_by_checkpoint = bdf.groupby("checkpoint")[["parameters", "gradients", "optimizer_state"]].mean()
print("Average category sizes (GB) by checkpoint:")
avg_by_checkpoint

In [ ]:
ax = avg_by_checkpoint.plot(kind="bar", figsize=(7, 4))
ax.set_ylabel("GB")
ax.set_title("Category breakdown at start-of-update vs peak-of-update")
plt.tight_layout()
plt.show()

## 5. Idle-time characterization (Experiment 3)

Simplifying proxy: in a standard sequential PPO loop, optimizer state is untouched during the rollout phase (nothing calls `optimizer.step()` while generating), and the generation-time KV cache is not reused during the update phase. Under that assumption, each category's idle time within a step is just the wall-clock time spent in the *other* phase.

In [ ]:
phase_time = df.groupby("phase")["t"].count() * poller.interval_s  # sample count * interval ≈ wall time
total_time = phase_time.sum()

rollout_time = phase_time.get("rollout", 0.0)
update_time = phase_time.get("update", 0.0)

print(f"rollout phase: {rollout_time:.2f}s ({rollout_time / total_time:.1%} of total)")
print(f"update phase:  {update_time:.2f}s ({update_time / total_time:.1%} of total)")
print()
print(f"optimizer_state idle fraction (≈ rollout share): {rollout_time / total_time:.1%}")
print(f"rollout KV-cache idle fraction (≈ update share): {update_time / total_time:.1%}")

## Extensions (not run here)

- **Experiment 4 (workload sweep):** re-run section 2 with different `max_new_tokens` / batch sizes and compare the idle fractions above.
- **Experiment 5 (offload cost baseline):** wrap `optimizer` with DeepSpeed's ZeRO-Offload (or a manual `.to("cpu", non_blocking=True)` round-trip on the optimizer state) and re-measure `update_time` to quantify the added latency.
- **Track B (veRL):** see `notebooks/02_verl_phase_analysis.ipynb`, which analyzes traces produced by `scripts/memory_poller.py --track verl` run on a GPU box (veRL's Ray/vLLM orchestration doesn't run cleanly inside a hosted notebook).

## References

1. Sheng, G. et al. "HybridFlow: A Flexible and Efficient RLHF Framework." EuroSys 2025. [arXiv:2409.19256](https://arxiv.org/abs/2409.19256)
2. Wang, X. et al. "LlamaRL: A Distributed Asynchronous RL Framework for Efficient Large-scale LLM Training." [arXiv:2505.24034](https://arxiv.org/abs/2505.24034)
3. Ren, J. et al. "ZeRO-Offload: Democratizing Billion-Scale Model Training." [arXiv:2101.06840](https://arxiv.org/abs/2101.06840)
4. Lan, T. et al. "ZenFlow: Enabling Stall-Free Offloading Training via Asynchronous Updates." [arXiv:2505.12242](https://arxiv.org/abs/2505.12242)
5. "Rollplex: Cross-Phase GPU Spatial Sharing for Vision Language Model Post-Training." [arXiv:2608.14498](https://arxiv.org/abs/2608.14498)
6. Feng, Y. et al. "Ada-KV: Optimizing KV Cache Eviction by Adaptive Budget Allocation." [arXiv:2407.11550](https://arxiv.org/abs/2407.11550)
7. Wang, X. et al. "Exploring and Evaluating Real-world CXL: Use Cases and System Adoption." IPDPS 2025. [arXiv:2405.14209](https://arxiv.org/abs/2405.14209)